## Análisis de rendimiento: definición de "temporada buena vs mala"

Este notebook parte de `df_maestro.csv`, ya construido y validado en `01_exploracion_estructura.ipynb`. Aquí se define una métrica compuesta de rendimiento por temporada y se realiza el análisis exploratorio de la relación entre gasto, entrenador y rendimiento.

### Decisión de diseño: qué define una "temporada buena"

El fútbol es un deporte resultadista: el objetivo de cada temporada no es alcanzar una cifra específica de puntos, es terminar por delante del resto de la liga. Por eso, la categoría de éxito/fracaso de una temporada (`categoria_temporada`) se define exclusivamente por **posición final**, no por puntos:

- **Título**: posición 1
- **Top 4**: posición 2 a 4 (zona de clasificación a Champions League)
- **Mitad de tabla**: posición 5 a 10
- **Mala**: posición 11 en adelante

En paralelo, se calcula `rendimiento_relativo`, una variable continua (puntos obtenidos sobre el máximo de puntos posible esa temporada específica). Esta normalización es necesaria porque los puntos crudos no son comparables entre eras: la regla de puntos por victoria cambió de 2 a 3 en 1995/96, y dos temporadas (1995/96 y 1996/97) tuvieron 42 partidos en vez de 38. `rendimiento_relativo` corrige ambas diferencias sin modificar el dato base de puntos, que se mantiene fiel a la clasificación real de cada año.

**Uso previsto de cada variable:** `categoria_temporada` se usa para comparaciones narrativas y visualizaciones (por ejemplo, cuántas temporadas de título tuvo cada entrenador). `rendimiento_relativo` se usa para análisis estadístico que requiere una variable numérica continua, como la correlación entre gasto neto y rendimiento. No se usa gasto ni entrenador como parte de la definición de "temporada buena", para evitar que el criterio de éxito quede contaminado por las mismas variables que después se van a analizar en relación a él.

In [4]:
import pandas as pd

df_maestro = pd.read_csv('../data/processed/df_maestro.csv', dtype={'temporada': str})
df_maestro.head()

,temporada,posicion,puntos,partidos_jugados,victorias,empates,derrotas,goles_favor,goles_contra,diferencia_gol,anio_inicio,entrenador_principal,hubo_cambio_entrenador,gasto_compras,ingreso_ventas,gasto_neto,datos_gasto_disponibles
0,9394,1,56,38,25,6,7,91,42,49,1993,Johan Cruyff,False,NaN,NaN,NaN,False
1,9495,4,46,38,18,10,10,60,45,15,1994,Johan Cruyff,False,NaN,NaN,NaN,False
2,9596,3,80,42,22,14,6,72,39,33,1995,Johan Cruyff,True,NaN,NaN,NaN,False
3,9697,2,90,42,28,6,8,102,48,54,1996,Bobby Robson,False,NaN,NaN,NaN,False
4,9798,1,74,38,23,5,10,78,56,22,1997,Louis van Gaal,False,NaN,NaN,NaN,False


In [5]:
df_maestro['puntos_por_victoria'] = df_maestro['temporada'].apply(lambda t: 2 if t in ['9394', '9495'] else 3)
df_maestro['puntos_maximos_posibles'] = df_maestro['partidos_jugados'] * df_maestro['puntos_por_victoria']
df_maestro['rendimiento_relativo'] = df_maestro['puntos'] / df_maestro['puntos_maximos_posibles']

df_maestro[['temporada', 'posicion', 'puntos', 'puntos_maximos_posibles', 'rendimiento_relativo']].head(10)

,temporada,posicion,puntos,puntos_maximos_posibles,rendimiento_relativo
0,9394,1,56,76,0.736842
1,9495,4,46,76,0.605263
2,9596,3,80,126,0.634921
3,9697,2,90,126,0.714286
4,9798,1,74,114,0.649123
5,9899,1,79,114,0.692982
6,9900,2,64,114,0.561404
7,0001,4,63,114,0.552632
8,0102,4,64,114,0.561404
9,0203,6,56,114,0.491228


In [6]:
def categorizar_temporada(posicion):
    if posicion == 1:
        return 'Título'
    elif posicion <= 4:
        return 'Top 4'
    elif posicion <= 10:
        return 'Mitad de tabla'
    else:
        return 'Mala'

df_maestro['categoria_temporada'] = df_maestro['posicion'].apply(categorizar_temporada)
df_maestro['categoria_temporada'].value_counts()

categoria_temporada
Título            16
Top 4             16
Mitad de tabla     1
Name: count, dtype: int64